# Stage 1 / Step 2 - Component 2: Topic alignment score (BERTopic)

"Is this topic currently resonating with users?" -> a score in [0, 1].

BERTopic clusters the ad text (title + description) into topics. A topic that many posts talk
about = a resonating / trending topic. `topic_score` = how 'trending' the post's topic is
(its share of the corpus). This is label-free (no leakage) and improves as the dataset grows.

In [ ]:
# load + fit BERTopic on the ad text
from pathlib import Path
import pandas as pd
import numpy as np
from bertopic import BERTopic

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_parquet(ROOT / "ml" / "data" / "video_features.parquet")

# 'ad text' = title + description (what a marketer writes pre-launch; cleaner topics than transcript)
vids = pd.read_csv(ROOT / "data" / "samples" / "video_context_dataset.csv")[["video_id", "video_title", "video_description"]]
vids = vids.fillna("")
df = df.merge(vids, on="video_id", how="left")
df["ad_text"] = (df["video_title"] + ". " + df["video_description"]).str.strip()

topic_model = BERTopic(embedding_model="all-MiniLM-L6-v2", min_topic_size=12, verbose=False)
topics, _ = topic_model.fit_transform(df["ad_text"].tolist())
df["topic"] = topics
print("num topics (excl. -1 outliers):", len([t for t in set(topics) if t != -1]))

In [ ]:
# inspect the discovered topics (interpretability for marketers)
info = topic_model.get_topic_info()[["Topic", "Count", "Name"]]
print(info.head(12).to_string(index=False))

In [ ]:
# topic_score = how 'trending' the post's topic is = topic share of corpus, normalized to [0,1]
sizes = df[df["topic"] != -1]["topic"].value_counts()
trend_weight = (sizes / sizes.max()).to_dict()          # biggest topic -> 1.0
df["topic_score"] = df["topic"].map(trend_weight).fillna(0.0)   # outliers (-1) -> 0
print(df["topic_score"].describe().round(3))

stage2 = df[["video_id", "topic", "topic_score"]]
stage2.to_parquet(ROOT / "ml" / "data" / "stage2_topic_scores.parquet", index=False)
print("Saved -> ml/data/stage2_topic_scores.parquet")